# Experiment 01: Autoencoder Baseline

**Audience**
- Readers who know basic PyTorch tensors and training loops but want intuition for latent bottlenecks before introducing variational inference.

**Prerequisites**
- You understand supervised optimization at a high level.
- You know what an encoder and decoder do.
- You can run notebook cells and inspect generated files.

**Learning goals**
- Understand what experiment 1 is measuring and why it comes before the VAE experiments.
- Inspect the exact model and dataset used by the shared experiment runner.
- Learn how to judge whether a deterministic bottleneck is doing useful work.
- Leave this notebook with a concrete baseline to compare against `exp_02_vanilla_vae_baseline`.


## Outline

1. Frame the question: what does a deterministic autoencoder teach us?
2. Preview the MNIST reconstruction task.
3. Inspect the exact architecture and bottleneck size.
4. Make a prediction before training.
5. Run the shared experiment code.
6. Read the saved metrics and artifacts.
7. Summarize what this baseline tells us before moving to VAEs.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from pprint import pprint

import torch
from IPython.display import Image, Markdown, display
from torchvision.utils import save_image

from sandbox_autoencoders.datasets.registry import get_dataset_adapter
from sandbox_autoencoders.experiments.shared import get_experiment_spec
from sandbox_autoencoders.models.conv import Autoencoder
from sandbox_autoencoders.training.runner import execute_experiment
from sandbox_autoencoders.utils.io import ensure_dir, seed_everything


def extract_images(batch):
    if isinstance(batch, dict):
        return batch["image"]
    if isinstance(batch, (list, tuple)):
        return batch[0]
    return batch


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def describe_active_latents(count: int, latent_dim: int) -> str:
    if count == 0:
        return "No latent dimension crossed the variance threshold in this run. In a very short run this can happen; in a longer run it is a warning sign that the bottleneck is underused."
    if count < max(2, latent_dim // 4):
        return "Only a small fraction of the bottleneck is active. The model may be reconstructing with limited capacity or the run may be too short to expose richer usage."
    if count < latent_dim:
        return "Several latent dimensions are active, which is healthy for this deterministic baseline. Not every dimension needs to be used equally."
    return "Every latent dimension crossed the activity threshold. That is strong evidence that the bottleneck is engaged rather than ignored."


seed_everything(7)
torch.__version__


## 1. Why Start With A Deterministic Autoencoder?

This experiment removes the VAE-specific complexity on purpose.

- The encoder compresses an image into a fixed latent vector.
- The decoder tries to reconstruct the original image from that vector.
- There is **no KL term** and **no learned prior** yet.

That makes experiment 1 the cleanest place to learn three things:

- Whether the convolutional encoder/decoder stack is capable of learning the dataset at all.
- What reconstruction quality looks like before we impose generative structure.
- How much latent activity we lose later when a VAE starts paying a KL penalty.

The key mindset: this notebook is not asking whether the model is generative. It is asking whether the bottleneck learns a useful compressed representation.

In [ ]:
EXPERIMENT_ID = "exp_01_autoencoder_baseline"
spec = get_experiment_spec(EXPERIMENT_ID)
variant = spec.variants_factory(spec.default_epochs)[0]

DATA_ROOT = Path("data")
OUTPUT_ROOT = Path("outputs/notebooks")
TMP_ROOT = ensure_dir(Path("tmp/jupyter-notebook"))
RUN_EPOCHS = spec.default_epochs
BATCH_SIZE = 32
SEED = 7
DEVICE = "cpu"
NUM_WORKERS = 0
WANDB_MODE = "disabled"
USE_SYNTHETIC_DATA = False  # Flip to True for a fast smoke-style run.

if USE_SYNTHETIC_DATA:
    os.environ["SANDBOX_AUTOENCODERS_TEST_MODE"] = "1"
else:
    os.environ.pop("SANDBOX_AUTOENCODERS_TEST_MODE", None)

variant.epochs = RUN_EPOCHS

run_config = {
    "description": spec.description,
    "dataset_id": variant.dataset_id,
    "model_type": variant.model_type,
    "latent_dim": variant.latent_dim,
    "loss_type": variant.loss_type,
    "decoder_variant": variant.decoder_variant,
    "epochs": variant.epochs,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "device": DEVICE,
    "wandb_mode": WANDB_MODE,
    "test_mode": USE_SYNTHETIC_DATA,
}

pprint(run_config)


## 2. Preview The Reconstruction Task

Before training, look at the inputs. MNIST is simple enough that we expect a small convolutional autoencoder to learn useful reconstructions quickly. That makes it a good first checkpoint for the shared training code and the bottleneck itself.


In [ ]:
adapter = get_dataset_adapter(variant.dataset_id)
bundle = adapter.build(data_root=DATA_ROOT, batch_size=BATCH_SIZE, seed=SEED, num_workers=NUM_WORKERS)
preview_batch = next(iter(bundle.val_loader))
preview_images = extract_images(preview_batch)

preview_path = TMP_ROOT / "exp01-mnist-preview.png"
save_image(preview_images[:16], preview_path, nrow=4)

display(
    Markdown(
        f"Dataset: `{bundle.dataset_id}`  \n"
        f"Sample shape: `{bundle.sample_shape}`  \n"
        f"Normalization metadata: `{bundle.normalization}`"
    )
)
display(Image(filename=str(preview_path)))


What you should notice:

- Inputs are grayscale and low-resolution, so reconstruction difficulty is moderate rather than extreme.
- Digits share coarse structure, which should help the model learn quickly.
- Fine details still matter: if the bottleneck is too weak, digits become average-looking blobs rather than distinct shapes.


## 3. Inspect The Exact Architecture

This cell instantiates the same deterministic model family used by the experiment runner. The important idea is not just that there is an encoder and decoder, but that a 16-dimensional latent vector must carry enough information to reconstruct 32x32 digits.


In [ ]:
model = Autoencoder(
    in_channels=bundle.sample_shape[0],
    image_size=bundle.sample_shape[1],
    latent_dim=variant.latent_dim,
    decoder_variant=variant.decoder_variant,
)

param_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

with torch.no_grad():
    sample_output = model(preview_images[:8])

model_summary = {
    "parameter_count": param_count,
    "trainable_parameters": trainable_count,
    "input_shape": tuple(preview_images[:8].shape),
    "latent_shape": tuple(sample_output.latent.shape),
    "reconstruction_shape": tuple(sample_output.recon.shape),
}

pprint(model_summary)
print(model)


Interpretation:

- `latent_shape = (batch, 16)` means every image must be summarized in only 16 numbers.
- If reconstructions still look good after training, the encoder has learned a compact representation rather than memorizing pixels directly.
- Later, the VAE will use a similar backbone but add distribution parameters and a KL penalty. This baseline tells us what quality we are giving up for that extra structure.


## 4. Predict Before You Train

Pause here and make a falsifiable prediction. This matters for learning: if you write down what you expect, the result teaches you something either way.


In [ ]:
prediction_notes = {
    "will_val_recon_loss_drop": "yes",
    "how_sharp_will_reconstructions_be": "sharper than random, probably quite good by the end of the run",
    "how_many_latents_will_be_active": "several, but not necessarily all 16",
    "biggest_risk": "blurry reconstructions if the bottleneck is too constrained or the run is too short",
}

prediction_notes


## 5. Run The Shared Experiment Code

This is the same path used by the CLI module. The point of the notebook is to make the workflow inspectable, not to fork the experiment implementation.


In [ ]:
combined_summary = execute_experiment(
    experiment_id=spec.experiment_id,
    variants=[variant],
    data_root=DATA_ROOT,
    output_dir=OUTPUT_ROOT,
    batch_size=BATCH_SIZE,
    seed=SEED,
    device_name=DEVICE,
    num_workers=NUM_WORKERS,
    wandb_mode=WANDB_MODE,
    wandb_project="vae-curriculum",
    wandb_entity=None,
    wandb_run_name=None,
    wandb_group=None,
    wandb_tags=["notebook", "exp01", "learning"],
)

combined_summary


## 6. Read The Saved Outputs

The training loop writes canonical artifacts to disk. Read those files instead of relying on notebook-local state so your interpretation matches what the CLI run would produce.


In [ ]:
experiment_dir = OUTPUT_ROOT / EXPERIMENT_ID
variant_dir = experiment_dir / variant.name

history = load_json(variant_dir / "history.json")
variant_summary = load_json(variant_dir / "summary.json")
latent_stats = load_json(variant_dir / "latent_stats.json")
notes = (variant_dir / "notes.md").read_text()

artifact_overview = {
    "experiment_dir": str(experiment_dir),
    "variant_dir": str(variant_dir),
    "final_epoch": history[-1],
    "summary": variant_summary,
    "active_latents_note": describe_active_latents(variant_summary["active_latents"], variant.latent_dim),
}

pprint(artifact_overview)


In [ ]:
epochs = [row["epoch"] for row in history]

try:
    import matplotlib.pyplot as plt
except ImportError:
    compact_history = [
        {
            "epoch": row["epoch"],
            "train_loss": round(row["train_loss"], 4),
            "val_loss": round(row["val_loss"], 4),
            "val_recon_loss": round(row["val_recon_loss"], 4),
            "active_latents": row["active_latents"],
        }
        for row in history
    ]
    pprint(compact_history)
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], marker="o", label="train_loss")
    axes[0].plot(epochs, [row["val_loss"] for row in history], marker="o", label="val_loss")
    axes[0].plot(epochs, [row["val_recon_loss"] for row in history], marker="o", label="val_recon_loss")
    axes[0].set_title("Losses")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, [row["active_latents"] for row in history], marker="o", color="tab:green")
    axes[1].axhline(variant.latent_dim, linestyle="--", color="tab:gray", linewidth=1)
    axes[1].set_title("Active Latents")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.show()


How to read these outputs:

- Falling `val_recon_loss` means the model is learning to preserve more input detail through the bottleneck.
- `active_latents` is a simple variance-based diagnostic, not a perfect truth metric. It tells you whether dimensions are being used noticeably, not whether they are semantically clean.
- In this deterministic baseline, `train_kl_loss` and `val_kl_loss` stay at zero by design. That is not a bug here; it is exactly what makes this experiment a clean starting point.


In [ ]:
display(Markdown(notes))

display(Markdown("### Reconstruction Grid"))
display(Image(filename=str(variant_dir / "recon_grid.png")))

interp_path = variant_dir / "interp_sheet.png"
if interp_path.exists():
    display(Markdown("### Latent Interpolation"))
    display(Image(filename=str(interp_path)))


## 7. Guided Interpretation

Use the run outputs to answer these questions:

1. Did the validation reconstruction loss improve materially from the first epoch to the last?
2. Do the reconstructions preserve digit identity, or do they average away important strokes?
3. Is the bottleneck clearly engaged, or is the model relying on only a tiny slice of its 16-dimensional capacity?
4. If interpolation looks smooth, does that mean the model is generative? No. It only means the decoder behaves reasonably between two encoded points.


In [ ]:
recon_delta = history[0]["val_recon_loss"] - history[-1]["val_recon_loss"]

learning_summary = {
    "prediction_check": {
        "predicted_recon_loss_drop": prediction_notes["will_val_recon_loss_drop"],
        "observed_recon_loss_drop": round(recon_delta, 6),
    },
    "final_active_latents": variant_summary["active_latents"],
    "active_latents_interpretation": describe_active_latents(variant_summary["active_latents"], variant.latent_dim),
    "baseline_takeaway": "Use this run as the reconstruction-first reference point before introducing the KL tradeoff in experiment 2.",
    "my_notes": "Replace this string with your own summary after inspecting the images.",
}

pprint(learning_summary)


## Exercise

Change exactly one variable, rerun, and write down what changed.

Good first choices:
- Reduce `variant.latent_dim` to `4`.
- Increase `variant.latent_dim` to `64`.
- Increase `RUN_EPOCHS` and compare whether more latent dimensions become active.

The point is to connect the bottleneck size to visible reconstruction behavior, not just to collect a lower loss number.


In [ ]:
exercise_log = {
    "change_i_made": "",
    "prediction_before_rerun": "",
    "what_changed_in_reconstructions": "",
    "what_changed_in_active_latents": "",
    "what_i_learned": "",
}

exercise_log


## Pitfalls And Extensions

**Common mistake**
- Treating smooth interpolation as proof that the model has learned a valid generative latent space. In experiment 1 there is no prior pressure at all, so interpolation quality is only a local decoder sanity check.

**Extension**
- Open experiment 2 next and compare this notebook's reconstruction sharpness against a VAE run with nonzero KL. That contrast is the core lesson of the curriculum.
